# K-Means Baseline

This notebook creates the first K-Means clustering baseline for the customer segmentation project.

The goal is to test different numbers of clusters and compare simple evaluation metrics. This result is not necessarily the final clustering solution. The next phase will interpret and profile the clusters so the solution can be judged from a business perspective.

## Load Data

The modelling input is the selected feature table from the previous phase. It is already scaled, encoded, compact, and prepared for the first clustering baseline.

In [1]:
import os
import sys
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

if os.getcwd().endswith("notebooks"):
    os.chdir("..")

sys.path.append(os.getcwd())


from src.clustering import (
    calculate_clustering_metrics,
    get_cluster_summary,
    save_cluster_assignments,
    split_customer_features,
    validate_clustering_input,
)

os.makedirs("outputs/clustering", exist_ok=True)
os.makedirs("outputs/figures", exist_ok=True)

df = pd.read_csv("data/processed/selected_model_features.csv")
print(f"Loaded rows: {len(df):,}")
print(f"Loaded columns: {df.shape[1]}")

Loaded rows: 33,038
Loaded columns: 26


## Data Validation

Before modelling, the notebook checks that every customer appears once, that there are no missing values, and that all modelling columns are numeric.

In [2]:
validate_clustering_input(df)

row_count = len(df)
unique_customer_count = df["customer_id"].nunique()
duplicated_customer_count = df["customer_id"].duplicated().sum()
missing_value_count = df.isna().sum().sum()
non_numeric_columns = df.drop(columns=["customer_id"]).select_dtypes(exclude="number").columns.tolist()

print(f"Rows loaded: {row_count:,}")
print(f"Unique customer_id values: {unique_customer_count:,}")
print(f"Duplicated customer_id count: {duplicated_customer_count:,}")
print(f"Total missing values: {missing_value_count:,}")
print(f"Modelling columns are numeric: {len(non_numeric_columns) == 0}")

print("Input validation passed.")

Rows loaded: 33,038
Unique customer_id values: 33,038
Duplicated customer_id count: 0
Total missing values: 0
Modelling columns are numeric: True
Input validation passed.


## Prepare Modelling Data

The `customer_id` column is kept only for saving the final assignments. It is not used as a K-Means feature.

In [3]:
customer_ids, X = split_customer_features(df)

print(f"Customers: {len(customer_ids):,}")
print(f"Model features: {X.shape[1]}")

Customers: 33,038
Model features: 25


## Test K-Means Solutions

The baseline tests cluster counts from 2 to 10. The silhouette score uses a sample of up to 10,000 rows for performance.

In [4]:
k_values = range(2, 11)
metrics_rows = []

for k in k_values:
    print(f"Fitting K-Means with k={k}")
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=50)
    labels = kmeans.fit_predict(X)
    metric_values = calculate_clustering_metrics(X, labels)
    metric_values["k"] = k
    metric_values["inertia"] = kmeans.inertia_

    metrics_rows.append(metric_values)

metrics = pd.DataFrame(metrics_rows)
metrics = metrics[[
    "k",
    "inertia",
    "silhouette_score",
    "calinski_harabasz_score",
    "davies_bouldin_score",
    "min_cluster_size",
    "max_cluster_size",
    "min_cluster_percentage",
    "max_cluster_percentage",
]]
metrics.to_csv("outputs/clustering/kmeans_metrics.csv", index=False)

print("Saved metrics to outputs/clustering/kmeans_metrics.csv")
metrics

Fitting K-Means with k=2
Fitting K-Means with k=3
Fitting K-Means with k=4
Fitting K-Means with k=5
Fitting K-Means with k=6
Fitting K-Means with k=7
Fitting K-Means with k=8
Fitting K-Means with k=9
Fitting K-Means with k=10
Saved metrics to outputs/clustering/kmeans_metrics.csv


## Evaluation Charts

These charts compare compactness, separation, and cluster balance across the tested values of `k`.

In [5]:
plt.figure(figsize=(7, 4))
plt.plot(metrics["k"], metrics["inertia"], marker="o")
plt.xlabel("Number of clusters (k)")
plt.ylabel("Inertia")
plt.title("K-Means Elbow Curve")
plt.xticks(metrics["k"])
plt.tight_layout()
plt.savefig("outputs/figures/kmeans_elbow.png", dpi=150)
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(metrics["k"], metrics["silhouette_score"], marker="o")
plt.xlabel("Number of clusters (k)")
plt.ylabel("Silhouette score")
plt.title("K-Means Silhouette Scores")
plt.xticks(metrics["k"])
plt.tight_layout()
plt.savefig("outputs/figures/kmeans_silhouette.png", dpi=150)
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(metrics["k"], metrics["calinski_harabasz_score"], marker="o")
plt.xlabel("Number of clusters (k)")
plt.ylabel("Calinski-Harabasz score")
plt.title("K-Means Calinski-Harabasz Scores")
plt.xticks(metrics["k"])
plt.tight_layout()
plt.savefig("outputs/figures/kmeans_calinski_harabasz.png", dpi=150)
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(metrics["k"], metrics["davies_bouldin_score"], marker="o")
plt.xlabel("Number of clusters (k)")
plt.ylabel("Davies-Bouldin score")
plt.title("K-Means Davies-Bouldin Scores")
plt.xticks(metrics["k"])
plt.tight_layout()
plt.savefig("outputs/figures/kmeans_davies_bouldin.png", dpi=150)
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(metrics["k"], metrics["min_cluster_percentage"], marker="o", label="Smallest cluster")
plt.plot(metrics["k"], metrics["max_cluster_percentage"], marker="o", label="Largest cluster")
plt.xlabel("Number of clusters (k)")
plt.ylabel("Cluster share (%)")
plt.title("K-Means Cluster Balance")
plt.xticks(metrics["k"])
plt.legend()
plt.tight_layout()
plt.savefig("outputs/figures/kmeans_cluster_balance.png", dpi=150)
plt.show()

notebooks/05_kmeans_baseline.ipynb:cell-11:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
notebooks/05_kmeans_baseline.ipynb:cell-11:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
notebooks/05_kmeans_baseline.ipynb:cell-11:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
notebooks/05_kmeans_baseline.ipynb:cell-11:39: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
notebooks/05_kmeans_baseline.ipynb:cell-11:51: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


## Initial k Choice

The initial baseline solution uses `SELECTED_K = 5`.

`k=2` has the strongest silhouette score, but it places more than three quarters of customers into one cluster, so it is too broad for useful segmentation. `k=5` is a more balanced starting point: the elbow curve still shows meaningful improvement before the gains slow down, the silhouette score is the strongest option after `k=2`, the Davies-Bouldin score improves compared with `k=4`, and the cluster sizes remain interpretable.

This does not mean `k=5` is the final clustering solution. It is a baseline choice that must be profiled and interpreted in the next phase before deciding whether the segmentation is good enough for business use.

In [6]:
SELECTED_K = 5
print(f"Selected k: {SELECTED_K}")

Selected k: 5


## Final Baseline Model

The final baseline model is fitted on the selected modelling features, not on PCA components. The saved output contains each customer and their assigned cluster.

In [7]:
final_kmeans = KMeans(n_clusters=SELECTED_K, random_state=42, n_init=50)
final_labels = final_kmeans.fit_predict(X)

cluster_assignments = save_cluster_assignments(
    customer_ids,
    final_labels,
    "outputs/clustering/kmeans_customer_clusters.csv",
)

print("Saved cluster assignments to outputs/clustering/kmeans_customer_clusters.csv")
print(cluster_assignments.head())

Saved cluster assignments to outputs/clustering/kmeans_customer_clusters.csv
   customer_id  cluster
0            3        3
1            4        2
2            5        3
3            7        4
4            8        4


## PCA Visualization

PCA is used only for a two-dimensional visual inspection of the clusters. K-Means is not trained on the PCA components in this baseline.

In [8]:
pca = PCA(n_components=2)
pca_components = pca.fit_transform(X)
cluster_ticks = sorted(cluster_assignments["cluster"].unique())

plt.figure(figsize=(7, 5))
scatter = plt.scatter(
    pca_components[:, 0],
    pca_components[:, 1],
    c=final_labels,
    cmap="tab10",
    s=8,
    alpha=0.6,
)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("K-Means Clusters Viewed with PCA")
colorbar = plt.colorbar(scatter, ticks=cluster_ticks)
colorbar.set_label("Cluster")
plt.tight_layout()
plt.savefig("outputs/figures/kmeans_pca_clusters.png", dpi=150)
plt.show()

print(f"PCA explained variance ratio: {pca.explained_variance_ratio_}")

PCA explained variance ratio: [0.17950417 0.12414316]


notebooks/05_kmeans_baseline.ipynb:cell-17:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


## Final Validation

The final checks confirm that the saved cluster assignment file has one valid cluster label for every input customer.

In [9]:
saved_clusters = pd.read_csv("outputs/clustering/kmeans_customer_clusters.csv")
cluster_size_table = get_cluster_summary(saved_clusters["cluster"])
row_count_matches = len(saved_clusters) == len(df)

print(f"Rows in kmeans_customer_clusters.csv: {len(saved_clusters):,}")
print(f"Unique customer_id values: {saved_clusters['customer_id'].nunique():,}")
print(f"Missing cluster values: {saved_clusters['cluster'].isna().sum():,}")
print("Cluster size table:")
print(cluster_size_table.to_string(index=False))
print(f"Output row count matches input row count: {row_count_matches}")

if not row_count_matches:
    raise ValueError("Cluster output row count does not match the input row count.")

if saved_clusters["customer_id"].duplicated().sum() > 0:
    raise ValueError("Cluster output contains duplicated customer_id values.")

if saved_clusters["cluster"].isna().sum() > 0:
    raise ValueError("Cluster output contains missing cluster values.")

Rows in kmeans_customer_clusters.csv: 33,038
Unique customer_id values: 33,038
Missing cluster values: 0
Cluster size table:
 cluster  count  percentage
       0   2234    6.761911
       1  10841   32.813730
       2   6687   20.240329
       3   9889   29.932199
       4   3387   10.251831
Output row count matches input row count: True
